In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [4]:
dataTrain = pd.read_csv("train_1.csv")
for i in range(2, 11):
  currData = pd.read_csv(f"train_{i}.csv")
  dataTrain = pd.concat([dataTrain, currData], axis = 0)
# testFile = pd.read_csv(f"train_{i+1}.csv")
dataTrain

,target,smpl,id,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,...,feature_409,feature_410,feature_411,feature_412,feature_413,feature_414,feature_415,feature_416,feature_417,feature_418
0,0,train,0,0.468142,-1.045346,0.0,0.384487,0.435121,-1.178548,0.124543,...,-0.361507,-1.026853,0.0,1.418600,-0.929668,1.284014,0.731842,0.801786,-0.728297,-0.412095
1,0,train,1,-0.760983,0.515132,0.0,-1.673905,-0.393862,-1.584207,-0.439778,...,-0.546275,-1.489542,0.0,-0.622007,-0.473156,0.780020,0.648577,0.646100,-0.789362,0.083349
2,0,train,2,1.658855,0.915052,0.0,-0.581082,0.477199,-0.622226,0.390642,...,-0.485999,0.586012,0.0,0.361481,-0.364566,-1.318596,-0.385155,0.140133,0.123245,-0.670030
3,0,train,3,-0.638854,0.314099,0.0,0.000919,1.102342,-0.807371,0.329158,...,0.321985,-0.075827,0.0,-1.629672,0.876864,0.411271,0.433440,0.997364,2.829590,-1.275588
4,0,train,4,-1.091376,0.859811,0.0,-0.505439,1.665086,-0.912464,-0.332054,...,0.828886,0.140387,0.0,-0.624304,-2.197691,-1.479267,-0.465917,-0.014757,-0.320434,-0.511896
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3869,0,train,38726,-1.444262,0.134778,1.0,-0.873376,-0.086289,-1.865676,-0.920675,...,1.803028,-1.233545,2.0,-1.122653,1.136513,-1.577056,-1.607065,-0.356970,-0.311724,-0.310598
3870,0,train,38727,0.092018,0.432922,1.0,-0.658572,-0.921391,-0.366676,-0.580566,...,1.443324,-0.144798,0.0,0.768462,-0.092746,-0.376226,-1.070219,-0.403162,0.577551,-0.381662
3871,0,train,38728,0.286276,1.332479,0.0,0.712252,0.615988,0.294629,0.202283,...,0.263073,-0.675841,0.0,0.170975,1.864763,0.531987,-0.121067,0.603182,0.841980,0.538176
3872,0,train,38729,-0.172293,-0.185632,1.0,2.070378,-1.513238,1.026286,-2.067229,...,-0.693392,0.498243,0.0,0.411256,2.201532,0.506141,0.267898,1.071174,-0.102825,-0.144410


In [6]:
target = dataTrain["target"]
dataTrain = dataTrain.drop(columns = [ 'smpl', 'id'])

In [43]:
correlation_threshold_target = 0.045  # минимальная корреляция с target, чтобы оставить признак
correlation_threshold_features = 0.9  # максимальная корреляция между признаками, чтобы оставить один из них

# Корреляционная матрица всего датасета
correlation_matrix = dataTrain.corr()

# Шаг 1: Отбор признаков с достаточной корреляцией с целевой переменной 'target'
target_correlation = correlation_matrix["target"].abs()  # берем абсолютные значения корреляции с 'target'
selected_features = target_correlation[target_correlation > correlation_threshold_target].index.tolist()

# Исключаем 'target' из списка отобранных признаков
selected_features.remove('target')

# Шаг 2: Убираем признаки, которые сильно коррелируют друг с другом
filtered_features = selected_features.copy()
for i in range(len(selected_features)):
    for j in range(i + 1, len(selected_features)):
        # Если корреляция между признаками выше порога, убираем один из них
        if abs(correlation_matrix[selected_features[i]][selected_features[j]]) > correlation_threshold_features:
            if selected_features[j] in filtered_features:
                filtered_features.remove(selected_features[j])

# Новый датафрейм с отобранными признаками
dataTrain_filtered = dataTrain[filtered_features + ['target']]

print("Изначальное количество признаков:", dataTrain.shape[1])
print("Количество признаков после отбора:", dataTrain_filtered.shape[1])


Изначальное количество признаков: 419
Количество признаков после отбора: 57


In [39]:
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostClassifier
import numpy as np

# Разделение данных на признаки и целевую переменную
X = dataTrain_filtered.drop(columns=['target'])
y = dataTrain_filtered['target']

# Функция для очистки выбросов через квантильный метод
def remove_outliers(X, lower_quantile=0.05, upper_quantile=0.95):
    # Определяем границы квантилей для каждого столбца
    lower_bound = X.quantile(lower_quantile)
    upper_bound = X.quantile(upper_quantile)
    
    # Обрабатываем каждый столбец отдельно, чтобы гарантировать, что мы сравниваем значения для каждого признака
    X_cleaned = X.copy()
    for column in X.columns:
        X_cleaned[column] = X[column].clip(lower=lower_bound[column], upper=upper_bound[column])
    
    return X_cleaned

# Очистка выбросов
X = remove_outliers(X)

# Разделение данных на тренировочный, тестовый и валидационный наборы
xTrain, xTest, yTrain, yTest = train_test_split(X, y, test_size=0.2, random_state=42)
xTest, xVal, yTest, yVal = train_test_split(xTest, yTest, test_size=0.5, random_state=42)

# Нормализация данных
scaler = StandardScaler()
xTrain = scaler.fit_transform(xTrain)
xTest = scaler.transform(xTest)
xVal = scaler.transform(xVal)

# Определение функции оптимизации для Optuna
def objective(trial):
    # Задание гиперпараметров для CatBoost
    param = {
        'iterations': trial.suggest_int('iterations', 50, 300),
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.5, 1.0),
        'l2_leaf_reg': trial.suggest_loguniform('l2_leaf_reg', 1e-8, 1.0),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 10),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'random_seed': 42
    }

    # Обучение модели CatBoost с текущими параметрами
    model = CatBoostClassifier(**param, cat_features=[], verbose=0)  # CatBoost не требует указания категориальных признаков, если их нет
    model.fit(xTrain, yTrain)

    # Предсказание вероятностей на валидационном наборе и расчет ROC-AUC
    yVal_proba = model.predict_proba(xVal)[:, 1]
    roc_auc = roc_auc_score(yVal, yVal_proba)
    return roc_auc

# Настройка Optuna для оптимизации ROC-AUC
study = optuna.create_study(direction='maximize')

# n_trials - количество попыток (экспериментов) для поиска наилучших гиперпараметров
# Большее количество экспериментов повысит вероятность нахождения оптимальных гиперпараметров, но также увеличит время вычислений
study.optimize(objective, n_trials=50)

# Лучшие гиперпараметры
print("Наилучшие гиперпараметры:", study.best_params)

# Обучение модели с лучшими гиперпараметрами на всем тренировочном наборе
best_params = study.best_params
best_model = CatBoostClassifier(**best_params, random_seed=42, verbose=0)
best_model.fit(xTrain, yTrain)

# Предсказания и расчет ROC-AUC на тестовом наборе
yTest_proba = best_model.predict_proba(xTest)[:, 1]
roc_auc_test = roc_auc_score(yTest, yTest_proba)

print(f"ROC-AUC на тестовом наборе с лучшими гиперпараметрами: {roc_auc_test:.4f}")


[I 2024-11-10 03:51:20,784] A new study created in memory with name: no-name-c9d0a2dc-45dc-4b31-ae07-fcc804f0f70d
C:\Users\user\AppData\Local\Temp\ipykernel_11960\1589873359.py:44: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3),
C:\Users\user\AppData\Local\Temp\ipykernel_11960\1589873359.py:47: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'l2_leaf_reg': trial.suggest_loguniform('l2_leaf_reg', 1e-8, 1.0),
[I 2024-11-10 03:51:21,968] Trial 0 finished with value: 0.7329081691306647 and parameters: {'iterations': 67, 'depth': 8, 'learning_rate': 0.0770319871158493, 'subsample': 0.742538511734407, 'colsamp

Наилучшие гиперпараметры: {'iterations': 250, 'depth': 9, 'learning_rate': 0.04028433907656072, 'subsample': 0.5024912339432226, 'colsample_bylevel': 0.8123381210477807, 'l2_leaf_reg': 0.0021068251707643615, 'min_data_in_leaf': 7, 'border_count': 250}
ROC-AUC на тестовом наборе с лучшими гиперпараметрами: 0.7734


In [50]:
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import pandas as pd

# Функция для очистки выбросов через квантильный метод
def remove_outliers(X, lower_quantile=0.03, upper_quantile=0.97):
    # Определяем границы квантилей для каждого столбца
    lower_bound = X.quantile(lower_quantile)
    upper_bound = X.quantile(upper_quantile)
    
    # Обрабатываем каждый столбец отдельно, чтобы гарантировать, что мы сравниваем значения для каждого признака
    X_cleaned = X.copy()
    for column in X.columns:
        X_cleaned[column] = X[column].clip(lower=lower_bound[column], upper=upper_bound[column])
    
    return X_cleaned

# Разделение данных на признаки и целевую переменную
X = dataTrain_filtered.drop(columns=['target'])
y = dataTrain_filtered['target']

# Разделение данных на тренировочный, тестовый и валидационный наборы
xTrain, xTest, yTrain, yTest = train_test_split(X, y, test_size=0.2, random_state=42)
xTest, xVal, yTest, yVal = train_test_split(xTest, yTest, test_size=0.5, random_state=42)

# Очистка выбросов только в тренировочной выборке
xTrain = remove_outliers(xTrain)

# Нормализация данных (стандартизация)
scaler = StandardScaler()
xTrain = scaler.fit_transform(xTrain)
xTest = scaler.transform(xTest)
xVal = scaler.transform(xVal)

# Определение функции оптимизации для Optuna
def objective(trial):
    # Задание гиперпараметров для XGBoost
    param = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'use_label_encoder': False,
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 1.0),
        'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-8, 1.0),
        'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-8, 1.0),
        'gamma': trial.suggest_loguniform('gamma', 1e-8, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10)
    }

    # Обучение модели XGBoost с текущими параметрами
    model = XGBClassifier(**param, random_state=42)
    model.fit(xTrain, yTrain)

    # Предсказание вероятностей на валидационном наборе и расчет ROC-AUC
    yVal_proba = model.predict_proba(xVal)[:, 1]
    roc_auc = roc_auc_score(yVal, yVal_proba)
    return roc_auc

# Настройка Optuna для оптимизации ROC-AUC
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

# Лучшие гиперпараметры
print("Наилучшие гиперпараметры:", study.best_params)

# Обучение модели с лучшими гиперпараметрами на всем тренировочном наборе
best_params = study.best_params
best_model = XGBClassifier(**best_params, random_state=42)
best_model.fit(xTrain, yTrain)

# Предсказания и расчет ROC-AUC на тестовом наборе
yTest_proba = best_model.predict_proba(xTest)[:, 1]
roc_auc_test = roc_auc_score(yTest, yTest_proba)

print(f"ROC-AUC на тестовом наборе с лучшими гиперпараметрами: {roc_auc_test:.4f}")


[I 2024-11-10 04:14:10,350] A new study created in memory with name: no-name-bb9b36f3-ae20-4068-9b4f-6e0501e1eff2
C:\Users\user\AppData\Local\Temp\ipykernel_11960\3431262041.py:47: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3),
C:\Users\user\AppData\Local\Temp\ipykernel_11960\3431262041.py:50: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-8, 1.0),
C:\Users\user\AppData\Local\Temp\ipykernel_11960\3431262041.py:51: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/op

Наилучшие гиперпараметры: {'n_estimators': 74, 'max_depth': 7, 'learning_rate': 0.14180361691607754, 'subsample': 0.9493922369576575, 'colsample_bytree': 0.7003926557120648, 'reg_alpha': 0.0012507027430521025, 'reg_lambda': 2.0578432314770386e-06, 'gamma': 2.530958401149786e-07, 'min_child_weight': 1}
ROC-AUC на тестовом наборе с лучшими гиперпараметрами: 0.8012


In [45]:
# Лучшие гиперпараметры
print("Наилучшие гиперпараметры:", study.best_params)

print()

# Обучение модели с лучшими гиперпараметрами на всем тренировочном наборе
best_params = study.best_params
best_model = XGBClassifier(**best_params, random_state=42)
best_model.fit(xTrain, yTrain)

yTrain_proba = best_model.predict_proba(xTrain)[:, 1]
roc_auc_test = roc_auc_score(yTrain, yTrain_proba)

print(f"ROC-AUC на тренировочном наборе с лучшими гиперпараметрами: {roc_auc_test:.4f}")

yVal_proba = best_model.predict_proba(xVal)[:, 1]
roc_auc_test = roc_auc_score(yVal, yVal_proba)

print(f"ROC-AUC на валидационном наборе с лучшими гиперпараметрами: {roc_auc_test:.4f}")

# Предсказания и расчет ROC-AUC на тестовом наборе
yTest_proba = best_model.predict_proba(xTest)[:, 1]
roc_auc_test = roc_auc_score(yTest, yTest_proba)

print(f"ROC-AUC на тестовом наборе с лучшими гиперпараметрами: {roc_auc_test:.4f}")

Наилучшие гиперпараметры: {'n_estimators': 237, 'max_depth': 10, 'learning_rate': 0.030433549981454606, 'subsample': 0.95109680504395, 'colsample_bytree': 0.9754644750723871, 'reg_alpha': 0.019906354184658262, 'reg_lambda': 7.213662812025642e-06, 'gamma': 9.152573659439537e-08, 'min_child_weight': 1}

ROC-AUC на тренировочном наборе с лучшими гиперпараметрами: 1.0000
ROC-AUC на валидационном наборе с лучшими гиперпараметрами: 0.8071
ROC-AUC на тестовом наборе с лучшими гиперпараметрами: 0.8093
